# 05 — Compare, benchmark, and archetype APIs

anchor-op's downstream APIs let you (a) compare a measured operator against an externally-inferred
one, (b) run a benchmark against declared operator-level nulls, and (c) fit operator archetypes
across multiple cell-state measurements.

In [ ]:
import numpy as np
import anchorop as ao
rng = np.random.default_rng(0)

## Compare API

`ao.compare` takes a measured operator and a dict of externally-inferred operators (in the same
program coordinates) and returns a table of pairwise agreement metrics against optional nulls.

In [ ]:
d, n = 6, 30
J_true = (rng.normal(size=(d, d)) / np.sqrt(d)) - 2 * np.eye(d)
Wd = rng.normal(size=(d, n)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True)
kappa = 0.3 + 0.6 * rng.uniform(size=n)
U = -kappa[None, :] * Wd
S = -np.linalg.solve(J_true, U) + 0.02 * rng.normal(size=(d, n))
names = [f"g{i}" for i in range(n)]; effs = {nm: float(kappa[i]) for i, nm in enumerate(names)}
measurement = ao.measure_from_sensitivity(S, U, guide_names=names, guide_efficiencies=effs,
                                           reg="tsvd", reg_param="path", rank_tol=1e-2)

# Three "inferred" operators to compare (simulated methods)
J_exact = J_true.copy()                             # perfect
J_noisy = J_true + 0.2 * rng.normal(size=(d, d))    # noisy topology
J_sym   = 0.5 * (J_true + J_true.T)                 # symmetric part only (loses antisym)

results = ao.compare(
    measurement,
    {"exact_method": J_exact, "noisy_topology": J_noisy, "symmetric_only": J_sym},
    nulls=("shuffled_edges", "random_init"),
    n_null_draws=20, null_seed=0,
)
table = ao.comparison_table(results)
print(table)

The table reports for each (method, null) pair: operator relative Frobenius error, spectral
abscissa difference, symmetric/antisymmetric decomposition, and comparison against the null.

## Spectral helpers

Individual spectral metrics are available directly:

In [ ]:
print(f"spectral abscissa diff (exact vs measured): "
      f"{ao.spectral_abscissa_difference(J_exact, measurement.J):.4f}")
print(f"hyperbolicity sign (measured): {ao.hyperbolicity_sign(measurement.J)}")
summary = ao.spectral_summary(measurement.J)
print(f"spectral summary: {summary}")

## Benchmark shortcut

`ao.analyses.benchmark_report` runs `compare` and produces publication-ready bar figures and CSVs
in one call. It's what the paper's §3.1 uses to generate Fig 1d–e.

In [ ]:
report = ao.analyses.benchmark_report(
    measurement,
    {"exact_method": J_exact, "noisy_topology": J_noisy, "symmetric_only": J_sym},
    n_null_draws=20,
    save_dir=None,   # or a path if you want figures on disk
)
print("benchmark_report keys:", list(report.keys()))
print(f"per-method summary rows: {len(report['table'])}")

## Archetype API

`ao.fit_archetypes` fits operator archetypes across multiple cell-state measurements. Each archetype
is a symmetric matrix in the same program-basis coordinates; each cell state is a simplex-weighted
combination of the archetypes.

Requires ≥ 2 (ideally ≥ 4) full-rank measurements. Here we synthesize two.

In [ ]:
# Two synthetic states sharing an archetype
state_A_meas = measurement
J_state_B = J_true + 0.3 * rng.normal(size=(d, d))
S_B = -np.linalg.solve(J_state_B, U) + 0.02 * rng.normal(size=(d, n))
state_B_meas = ao.measure_from_sensitivity(S_B, U, guide_names=names, guide_efficiencies=effs,
                                             reg="tsvd", reg_param="path", rank_tol=1e-2)

arch = ao.fit_archetypes({"A": state_A_meas, "B": state_B_meas}, k=1)
print(f"archetype k = {arch.k}")
print(f"archetype shapes: {arch.archetypes.shape}   weights: {arch.weights.shape}")

### Transfer test

`ao.transfer_test` fits archetypes on a source cell state and evaluates the fit on a held-out
target state. Small transfer / refit error ratio = the source vocabulary generalizes.

In [ ]:
tr = ao.transfer_test({"A": state_A_meas, "B": state_B_meas}, source="A", target="B", k=1)
print(f"transfer error: {tr.transfer_error:.4f}")
print(f"refit error:    {tr.refit_error:.4f}")
print(f"ratio:          {tr.transfer_error / max(tr.refit_error, 1e-9):.3f}  (small = source vocab transfers)")

## Full archetype report

`ao.analyses.archetype_report` combines the fits, transfer test, and standard figures.

In [ ]:
archetype_report = ao.analyses.archetype_report(
    {"A": state_A_meas, "B": state_B_meas},
    ks=(1, 2),
    save_dir=None,
)
print("archetype_report keys:", list(archetype_report.keys()))

## Next

- **06**: fitting a program basis from expression data + high-level `ao.analyses.*_report` shortcuts.